# Multi-Armed Bandits: Explore vs Exploit

The **multi-armed bandit (MAB)** is the cleanest setting for the explore/exploit
dilemma — a *one-state MDP*. We have $k$ arms (actions); pulling arm $a$ returns a
reward drawn from that arm's *unknown* distribution, whose mean is the **action
value**
$$Q(a) = \mathbb{E}[r \mid a].$$
We want to maximize cumulative reward over $T$ pulls.

### What is regret?

Regret is how we measure "doing well" without knowing the arms in advance. Let
$V^* = Q(a^*) = \max_a Q(a)$ be the value of the *best* arm. Every time we pull a
suboptimal arm $a$ we give up its **gap** $\Delta_a = V^* - Q(a)$. The **total
regret** after $T$ steps is the reward lost versus always pulling the best arm:
$$L_T = \sum_{t=1}^{T}\big(V^* - Q(a_t)\big) = \sum_{a}\mathbb{E}[N_T(a)]\,\Delta_a,$$
where $N_T(a)$ is the number of times arm $a$ was pulled. So **maximizing reward is
the same as minimizing regret**. A good algorithm keeps $N_T(a)$ small for high-gap
arms — but it doesn't know the gaps up front, so it must *learn* them while paying
to explore. Curves that keep rising linearly never stop paying; curves that flatten
have found the best arm.

We implement and compare four strategies from the lecture on **regret curves**:

| Strategy | Idea | Regret |
|---|---|---|
| **ε-greedy** | exploit best estimate, explore at random (this is DQN's rule, Day 2) | linear |
| **UCB1** | optimism: a Hoeffding confidence bonus | ≈ log T |
| **Thompson sampling** | posterior sampling (Beta–Bernoulli) | ≈ log T |
| **Gradient bandit** | softmax preferences + REINFORCE update (Day 3, one state) | sublinear |

The **Lai–Robbins** lower bound says $\mathcal{O}(\log T)$ regret is the best any
algorithm can achieve — UCB1 and Thompson both match it.

## Setup

Pure NumPy + Matplotlib — no environment library needed.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## The Bernoulli bandit

Each arm $a$ pays reward $1$ with probability $p_a$ (its **mean**, so $Q(a) = p_a$)
and $0$ otherwise. The `means` array below holds these true win-probabilities — the
quantities the agent does *not* know and must estimate from pulls. Because *we* know
them, we can measure **regret** exactly: pulling arm $a$ costs the gap
$\Delta_a = p_{\max} - p_a$ relative to the best arm $a^*$.

In [ ]:
class BernoulliBandit:
    """k arms; arm a pays 1 w.p. means[a], else 0."""
    def __init__(self, means):
        self.means = np.asarray(means, dtype=float)
        self.k = len(self.means)
        self.best_value = self.means.max()

    def pull(self, a):
        return 1.0 if rng.random() < self.means[a] else 0.0

    def gap(self, a):
        return self.best_value - self.means[a]     # instantaneous (expected) regret

## Base agent + ε-greedy

Every strategy keeps its own statistics and exposes two methods: `select()` returns
an arm, `update(a, r)` folds in the observed reward. ε-greedy keeps a **sample-mean**
estimate $\hat{Q}(a)$ and, with probability $\epsilon$, pulls a uniformly random arm
instead of the greedy one — exactly DQN's exploration rule. Constant $\epsilon$
means it never stops exploring duds, so its regret is **linear**.

In [ ]:
class BanditAgent:
    """Base class: maintains per-arm sample-mean estimates Q and counts N."""
    name = "base"
    def __init__(self, k):
        self.k = k
        self.Q = np.zeros(k)
        self.N = np.zeros(k)
        self.t = 0

    def update(self, a, r):
        self.t += 1
        self.N[a] += 1
        self.Q[a] += (r - self.Q[a]) / self.N[a]      # incremental sample mean

    def select(self):
        raise NotImplementedError


class EpsilonGreedy(BanditAgent):
    def __init__(self, k, epsilon=0.1):
        super().__init__(k)
        self.epsilon = epsilon
        self.name = f"eps-greedy (eps={epsilon})"

    def select(self):
        if rng.random() < self.epsilon:
            return rng.integers(self.k)
        return int(np.argmax(self.Q))

## UCB1 — optimism in the face of uncertainty

Be optimistic: add a **confidence bonus** $\hat{U}_t(a)$ that is large for
rarely-pulled arms and shrinks as we gather data, then act greedily by the
optimistic value. Hoeffding's inequality (failure probability $t^{-\alpha}$) turns
into exactly this bonus:
$$a_{t+1} = \arg\max_{a}\ \Big\{\hat{Q}_t(a) + \underbrace{\sqrt{\tfrac{\alpha\,\log t}{2\,N_t(a)}}}_{\hat{U}_t(a)}\Big\}.$$
Here $\hat{Q}_t(a)$ is the current sample-mean estimate and $N_t(a)$ the pull count —
matching the lecture's notation. This is a real high-probability bound, not a
heuristic, and it earns **logarithmic** regret (Auer et al., 2002), matching
Lai–Robbins.

In [ ]:
class UCB1(BanditAgent):
    def __init__(self, k, alpha=2.0):
        super().__init__(k)
        self.alpha = alpha
        self.name = "UCB1"

    def select(self):
        # pull each arm once first so N(a) > 0 and the bonus is defined
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return int(untried[0])
        # Hoeffding upper-confidence bonus: big when N(a) is small, shrinks like sqrt(log t / N)
        bonus = np.sqrt(self.alpha * np.log(self.t) / (2 * self.N))
        return int(np.argmax(self.Q + bonus))

## Thompson sampling — posterior sampling

Instead of a bonus, keep a **posterior** over each arm's mean and *sample* from it.
For Bernoulli rewards with a uniform prior, the posterior after $S$ successes and
$F$ failures is $\mathrm{Beta}(1+S,\,1+F)$. Draw one sample per arm and play the
argmax — arms we're unsure about occasionally sample high and get explored. This is
**probability matching** by construction, and it also matches the Lai–Robbins bound.

In [ ]:
class ThompsonBernoulli(BanditAgent):
    def __init__(self, k):
        super().__init__(k)
        self.S = np.zeros(k)   # successes per arm
        self.F = np.zeros(k)   # failures per arm
        self.name = "Thompson"

    def select(self):
        # sample a mean from each arm's Beta posterior, then act greedily on the samples
        theta = rng.beta(1 + self.S, 1 + self.F)
        return int(np.argmax(theta))

    def update(self, a, r):
        super().update(a, r)
        self.S[a] += r
        self.F[a] += (1 - r)

## Gradient bandit — softmax scores

Keep a **score** $s(a)$ per arm (the lecture's $s_a$), act by a softmax policy
$\pi(a) = e^{s(a)} / \sum_b e^{s(b)}$, and nudge the scores by stochastic gradient
ascent on expected reward, using the running average reward $\bar r_t$ as a baseline:
$$s_{t+1}(a) = s_t(a) + \alpha\,(r_t - \bar r_t)\,\big(\mathbf{1}_{a=a_t} - \pi_t(a)\big).$$
This is exactly the **REINFORCE** estimator from Day 3 — score-function gradient
$r\,\nabla\log\pi$ with a variance-reducing baseline — applied to a one-state MDP.

In [ ]:
class GradientBandit(BanditAgent):
    def __init__(self, k, alpha=0.1):
        super().__init__(k)
        self.scores = np.zeros(k)   # score parameters s(a) (lecture's s_a)
        self.alpha = alpha
        self.r_bar = 0.0            # running average reward (baseline)
        self.name = "gradient"

    def _pi(self):
        z = self.scores - self.scores.max()
        e = np.exp(z)
        return e / e.sum()

    def select(self):
        return int(rng.choice(self.k, p=self._pi()))

    def update(self, a, r):
        self.t += 1
        self.N[a] += 1
        self.r_bar += (r - self.r_bar) / self.t
        pi = self._pi()
        # score update: the REINFORCE gradient with baseline r_bar (chosen arm rises if r > r_bar)
        one_hot = np.zeros(self.k); one_hot[a] = 1.0
        self.scores += self.alpha * (r - self.r_bar) * (one_hot - pi)

## Run + measure regret

We run each agent for $T$ steps, accumulating the **expected regret** $\Delta_{a_t}$
at every pull, and average over many independent seeds to get a smooth curve.

In [ ]:
def run(agent_class, means, T=5000, n_runs=200, **agent_kwargs):
    """Average cumulative-regret curve over n_runs seeds.

    A FRESH agent is built for each run, so we pass the agent *class* (plus its
    hyperparameters) and construct it inside — `agent_class(k, **agent_kwargs)` —
    rather than reusing a single already-trained instance.
    """
    cum = np.zeros(T)
    for _ in range(n_runs):
        bandit = BernoulliBandit(means)
        agent = agent_class(bandit.k, **agent_kwargs)     # a new agent per run
        regret = 0.0
        for t in range(T):
            a = agent.select()
            r = bandit.pull(a)
            agent.update(a, r)
            regret += bandit.gap(a)                        # pay the gap of the pulled arm
            cum[t] += regret
    return cum / n_runs


# True arm means (win probabilities Q(a) = p_a). Chosen so there is ONE clearly-best
# arm (0.9) plus a spread of clearly-worse "duds": this makes the explore/exploit
# trade-off visible and gives clean regret separation over T steps. The agent never
# sees these numbers — it only sees 0/1 rewards.
means = [0.9, 0.6, 0.55, 0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]
T = 5000

curves = {
    "eps-greedy (0.1)": run(EpsilonGreedy, means, T=T, epsilon=0.1),
    "UCB1":             run(UCB1, means, T=T),
    "Thompson":         run(ThompsonBernoulli, means, T=T),
    "gradient":         run(GradientBandit, means, T=T, alpha=0.1),
}
for name, c in curves.items():
    print(f"{name:18s} final cumulative regret @T={T}: {c[-1]:7.1f}")

## Regret curves

A **linear** curve means the agent keeps paying for exploration forever; a
**flattening** (sublinear, ≈ $\log T$) curve means it has essentially found the best
arm. Watch ε-greedy stay linear while UCB1 and Thompson bend over.

In [ ]:
plt.figure(figsize=(9, 5))
for name, c in curves.items():
    plt.plot(c, label=name)
plt.plot(np.log(np.arange(1, T + 1)) * (curves["UCB1"][-1] / np.log(T)),
         "k--", alpha=0.4, label="~log T reference")
plt.xlabel("step t")
plt.ylabel("cumulative regret")
plt.title("Bandit strategies: cumulative regret (lower is better)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Takeaways

- **ε-greedy** (DQN's rule) explores duds at a constant rate ⇒ **linear regret**.
- **UCB1** and **Thompson** are *principled*: their regret grows only as $\log T$,
  matching the Lai–Robbins lower bound — they stop wasting pulls once confident.
- **Gradient bandit** is REINFORCE on a one-state MDP; it learns a good softmax
  policy without ever estimating confidence bounds.

Everything here *lifts* to full MDPs: UCB → count-based/pseudo-count bonuses,
Thompson → bootstrapped DQN (PSRL), gradient bandit → the entropy bonus in SAC
(Day 6). That bridge to **deep exploration** is the next lab (RND).